In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(rc={'text.usetex':True, 'font.family':'sans-serif', 'font.sans-serif':'Helvetica'})

from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist

In [ ]:
id_col = 'clone_id_1_5_3'

# Schisto (XLI2)

In [ ]:
fam = pd.read_csv('../large_files/XLI2_factor1_gfold_fam.csv')
fam.head()

In [ ]:
targets = ['SmPDE','SmAC2','SmCA','SmAP']
filter_cols = [f'gfold01_{t}-HS' for t in targets]
plot_cols = [f'enrich_{t}-HS' for t in targets] + ['enrich_whole']
N = 30

name_convert = {
    'SmPDE':'SmNPP5',
    'SmAC2':'SmTAChE',
    'SmCA':'SmCA',
    'SmAP':'SmAP'
}

real_binders = fam[fam[filter_cols].apply(lambda x: x.sort_values(ascending=True).tolist()[-2]<=0,axis=1)]

selected = []
for col in filter_cols:
    top = real_binders[real_binders[col]>0.5].sort_values(by=col,ascending=False).head(N)
    # selected.append(top.sort_values(by=col.replace('gfold01','enrich'),ascending=False))
    selected.append(top.sort_values(by='enrich_whole',ascending=False))
selected = pd.concat(selected,axis=0)

plt.figure(figsize=[5,9])
ax = sns.heatmap(selected[plot_cols],cmap='bwr',vmin=-6,vmax=6)
plt.xticks(np.arange(len(plot_cols))+0.5,[name_convert[t] for t in targets]+['whole'],fontsize=16)
ax.tick_params(left=False, bottom=True, top=False, right=False)
plt.yticks([])

plt.savefig('XLI2_heatmap.png',dpi=300,bbox_inches='tight')

In [ ]:
selected[[id_col,'CDR1','CDR2','CDR3']+plot_cols].to_csv('XLI2_selected_fams.csv')

In [ ]:
fam.loc[fam['CDR3']=='NTGAV',[id_col,'CDR1','CDR2','CDR3']+plot_cols]

# BoNT (JHY3)

In [ ]:
fam = pd.read_csv('../large_files/JYH3_factor1_gfold_fam.csv')
fam.head()

In [ ]:
def comp_str(row):
    s = np.sign(row)

    if np.any(s<=0):
        d1 = np.where(s<=0)[0][0]
        d2 = int(np.sum(s<=0))
        return f'{str(d1).zfill(2)}{d2}'
    else:
        return '999'

In [ ]:
fam['max_frag'] = fam[['enrich_HcA1-coat','enrich_LcA1-6H']].max(axis=1)

# comp_order = [2,3,4,11,5,6,7,8,9,10]
comp_order = [5,6,7,10,2,3,9,4,11,8]
comp_convert = {f'comp{c}':f'CG-{i+1}' for i,c in enumerate(comp_order)}
comps = [f'comp{i}' for i in comp_order]
filter_cols = [f'gfold01_ciBoNTA-JDA-{c}' for c in comps]
# frags = ['HcA1-coat','HcA3-O','HcA8-O','LcA1-6H','LcA3-6H','LcA4-6H','LcHnA2-6H','LcHnA7-6H']
frags = ['LcA1-6H','HcA1-coat']
ctrls = ['JDA','6H']
ctrl_filter = [f'gfold01_ctrl-{c}' for c in ctrls]
plot_cols = ['enrich_ctrl-JDA','enrich_ciBoNTA-JDA']+[f'enrich_ciBoNTA-JDA-{c}' for c in comps]+[f'enrich_{f}' for f in frags]+['enrich_ctrl-6H']
labels = ['$\\alpha$-BoNT ctrl','ciBoNTA']+[comp_convert[c] for c in comps]+['LCA','HcA']+['$\\alpha$-6H ctrl']

real_binders = fam[(fam[ctrl_filter].max(axis=1)<=0)&(fam['gfold01_ciBoNTA-JDA']>1)]

N = 30
selected = []
for col in filter_cols:
    top = real_binders[real_binders[col]<0].sort_values(by=col,ascending=True).head(N)
    # selected.append(top.sort_values(by=col.replace('gfold01','enrich'),ascending=True))
    selected.append(top.sort_values(by='max_frag',ascending=False))

top = real_binders[real_binders[filter_cols].min(axis=1)>0].sort_values(by='enrich_ciBoNTA-JDA',ascending=False).head(N)
selected.append(top)

selected = pd.concat(selected,axis=0).drop_duplicates(id_col)

selected['comp_str'] = selected[filter_cols].apply(comp_str,axis=1)
selected = selected.groupby('comp_str').apply(lambda x: x,include_groups=False)

plt.figure(figsize=[8,9])
ax = sns.heatmap(selected[plot_cols],cmap='bwr',vmin=-6,vmax=6)
ax.tick_params(left=False, bottom=True, top=False, right=False)
plt.xticks(np.arange(len(plot_cols))+0.5,labels,fontsize=16)
plt.yticks([])
plt.ylabel('')

plt.savefig('JYH3_heatmap.png',dpi=300,bbox_inches='tight')

In [ ]:
selected[[id_col,'CDR1','CDR2','CDR3']+plot_cols].to_csv('JYH3_selected_fams.csv')

# CoV (XCE1)

In [ ]:
fam = pd.read_csv('../large_files/XCE1_factor1_gfold_fam.csv')
fam.head()

In [ ]:
comp_convert ={f'comp{i}':f'CG-{i}' for i in range(1,5)}
comp_convert['comp7'] = 'CG-5'
comp_convert['comp8'] = 'CG-6'

# comps = [f'comp{i}' for i in [1,2,3,4,5,6,7,8]]
comps = [f'comp{i}' for i in [1,2,3,4,7,8]]
filter_cols = [f'gfold01_CoV2-{c}' for c in comps]
vars = ['2d','2k','2m','2-BQ1.1','2-B1.1.529','2-BA2','2-BA2.75','2-BA4','1']
plot_cols = ['enrich_ctrl']+['enrich_CoV2']+[f'enrich_CoV2-{c}' for c in comps]+['enrich_CoV2-mono']+[f'enrich_CoV{v}' for v in vars]
var_labels = [f'CoV-{'.'.join(v.split('-')[1:]).replace('B.','B-').replace('Q.','Q-')}' if '-' in v else f'CoV{v}' for v in vars]
labels = ['ctrl']+['CoV2']+[comp_convert[c] for c in comps]+['CoV2-mono']+var_labels

fam['var_max'] = fam[[f'enrich_CoV{var}' for var in vars]].max(axis=1)

real_binders = fam[(fam['enrich_ctrl']<=0)&(fam[['gfold01_CoV2','gfold01_CoV1']].max(axis=1)>1)&(fam['gfold01_CoV2']>0)]

N = 30
selected = []
for col in filter_cols:
    top = real_binders[real_binders[col]<=0].sort_values(by=col,ascending=True).head(N)
    # top = top[top['gfold01_CoV2']>0]
    # selected.append(top.sort_values(by=col.replace('gfold01','enrich'),ascending=True))
    selected.append(top.sort_values(by='var_max',ascending=False))

top = real_binders[real_binders[filter_cols].min(axis=1)>0].sort_values(by='var_max',ascending=False).head(N)
selected.append(top)

selected = pd.concat(selected,axis=0).drop_duplicates(id_col)

selected['comp_str'] = selected[filter_cols].apply(comp_str,axis=1)
selected = selected.groupby('comp_str').apply(lambda x: x,include_groups=False)

plt.figure(figsize=[8,9])
ax = sns.heatmap(selected[plot_cols],cmap='bwr',vmin=-6,vmax=6)
ax.tick_params(left=False, bottom=True, top=False, right=False)
plt.xticks(np.arange(len(plot_cols))+0.5,labels,fontsize=16)
plt.yticks([])
plt.ylabel('')

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=16)

plt.savefig('XCE1_heatmap.png',dpi=300,bbox_inches='tight')

In [ ]:
selected[[id_col,'CDR1','CDR2','CDR3']+plot_cols].to_csv('XCE1_selected_fams.csv')